In [ ]:
%cd /content

# remove old folder if it already exists
!rm -rf scperturbevalv2

# clone repo
!git clone https://github.com/nainika0402-creator/scperturbevalv2.git

# enter correct folder
%cd scperturbevalv2

# install package
%pip install -e .

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/norman.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("project")

os.listdir("project")

In [ ]:
# !find /content -name "norman.h5ad"



# Preprocessing Norman Data under Additive Split

In [ ]:
%%bash

for FOLD in 0 1 2 3 4; do
  python /content/scperturbevalv2/scripts/preprocess_norman_scdfm.py \
    --norman-h5ad "/content/scperturbevalv2/project/norman/norman.h5ad" \
    --out-dir "/content/drive/MyDrive/baseline_results/${FOLD}_preprocessed" \
    --split-method additive \
    --fold ${FOLD} \
    --split-file "/content/scperturbevalv2/project/norman/split_results.pkl"

  echo "Fold ${FOLD} done"
done

echo "All folds completed"

#  Baselines

In [ ]:
%%bash
set -euo pipefail

PROJECT_ROOT="/content/scperturbevalv2"
OUT_ROOT="/content/drive/MyDrive/baseline_results"

# options:
# control
# global_delta_additive
# linear
# one_layer_mlp
# latent_additive
# decoder_only

BASELINE="linear"

for FOLD in 0 1 2 3; do

  PRE_DIR="${OUT_ROOT}/${FOLD}_preprocessed"
  OUT_DIR="${OUT_ROOT}/${FOLD}_${BASELINE}"

  echo "Starting fold ${FOLD}..."

  python "${PROJECT_ROOT}/scripts/run_baselines.py" \
    --processed-dir "${PRE_DIR}" \
    --baseline "${BASELINE}" \
    --steps 5000 \
    --hidden-dim 1024 \
    --latent-dim 64 \
    --lr 1e-3 \
    --seed 42 \
    --out-dir "${OUT_DIR}"

  echo "Fold ${FOLD} baseline ${BASELINE} done"

done

echo "Completed baseline: ${BASELINE} for folds 0..4"

# Metrics

In [ ]:
%%bash
set -euo pipefail
OUT_ROOT="/content/drive/MyDrive/baseline_results"
BASELINE="linear"

for FOLD in 0 1 2 3; do
  REAL="${OUT_ROOT}/${FOLD}_${BASELINE}/${BASELINE}_real.h5ad"
  PRED="${OUT_ROOT}/${FOLD}_${BASELINE}/${BASELINE}_pred.h5ad"
  OUT="${OUT_ROOT}/${FOLD}_${BASELINE}/mset1.csv"

  python -m scPerturbEval.evaluations \
    --real "$REAL" --pred "$PRED" \
    --condition-column condition --control-label control \
    --space raw \
    --metrics root_mean_squared_error pearson_distance pcc_delta top_deg_recall \
    --deg-selection topn --top-n-degs 100 \
    --out "$OUT"
done

In [ ]:
%%bash
set -euo pipefail
OUT_ROOT="/content/drive/MyDrive/baseline_results"
BASELINE="linear"

for FOLD in 0 1 2 3; do
  REAL="${OUT_ROOT}/${FOLD}_${BASELINE}/${BASELINE}_real.h5ad"
  PRED="${OUT_ROOT}/${FOLD}_${BASELINE}/${BASELINE}_pred.h5ad"
  OUT="${OUT_ROOT}/${FOLD}_${BASELINE}/mset2.csv"

  python -m scPerturbEval.evaluations \
    --real "$REAL" --pred "$PRED" \
    --condition-column condition --control-label control \
    --space raw \
    --metrics deg_direction_agreement deg_spearman_lfc pds_cosine matrix_distance \
    --deg-selection topn --top-n-degs 100 \
    --out "$OUT"
done

In [ ]:
%%bash
set -euo pipefail
OUT_ROOT="/content/drive/MyDrive/baseline_results"
BASELINE="linear"

for FOLD in 0 1 2 3; do
  REAL="${OUT_ROOT}/${FOLD}_${BASELINE}/${BASELINE}_real.h5ad"
  PRED="${OUT_ROOT}/${FOLD}_${BASELINE}/${BASELINE}_pred.h5ad"
  OUT="${OUT_ROOT}/${FOLD}_${BASELINE}/mset3.csv"

  python -m scPerturbEval.evaluations \
    --real "$REAL" --pred "$PRED" \
    --condition-column condition --control-label control \
    --space raw \
    --metrics wmse pearson_delta_pert weighted_r2_delta pathway_nes_spearman \
    --pathway-gene-sets MSigDB_Hallmark_2020 \
    --pathway-top-k 10 \
    --pathway-reference perturbed_centroid \
    --out "$OUT"
done

In [ ]:
%%bash
set -euo pipefail
OUT_ROOT="/content/drive/MyDrive/baseline_results"
BASELINE="linear"

for FOLD in 0 1 2 3; do
  REAL="${OUT_ROOT}/${FOLD}_${BASELINE}/${BASELINE}_real.h5ad"
  PRED="${OUT_ROOT}/${FOLD}_${BASELINE}/${BASELINE}_pred.h5ad"
  OUT="${OUT_ROOT}/${FOLD}_${BASELINE}/mset4_raw.csv"

  python -m scPerturbEval.evaluations \
    --real "$REAL" --pred "$PRED" \
    --condition-column condition --control-label control \
    --space raw \
    --metrics pathway_topk_jaccard \
    --pathway-gene-sets MSigDB_Hallmark_2020 \
    --pathway-top-k 10 \
    --pathway-reference perturbed_centroid \
    --out "$OUT"
done

In [ ]:
%%bash
set -euo pipefail
OUT_ROOT="/content/drive/MyDrive/baseline_results"
BASELINE="linear"

for FOLD in 0 1 2 3; do
  REAL="${OUT_ROOT}/${FOLD}_${BASELINE}/${BASELINE}_real.h5ad"
  PRED="${OUT_ROOT}/${FOLD}_${BASELINE}/${BASELINE}_pred.h5ad"
  OUT="${OUT_ROOT}/${FOLD}_${BASELINE}/mset5_pca.csv"

  python -m scPerturbEval.evaluations \
    --real "$REAL" --pred "$PRED" \
    --condition-column condition \
    --space pca --n-components 50 \
    --metrics wasserstein mmd \
    --out "$OUT"
done

# Aggregating Metrics across 4 folds

In [ ]:
%%bash
set -euo pipefail
python - <<'PY'
import pandas as pd
from pathlib import Path

OUT_ROOT = Path("/content/drive/MyDrive/baseline_results")
FOLDS = [0,1,2,3]
BASELINE="linear"

fold_rows = []
for f in FOLDS:
    d = OUT_ROOT / f"{f}_{BASELINE}"
    files = ["mset1.csv","mset2.csv","mset3.csv","mset4_raw.csv","mset5_pca.csv"]
    s = None
    for fn in files:
        df = pd.read_csv(d / fn).iloc[0]
        s = df if s is None else s.combine_first(df)
    s["fold"] = f
    fold_rows.append(s)

all_df = pd.DataFrame(fold_rows)
all_out = OUT_ROOT / f"{BASELINE}_all_metrics_fold_rows.csv"
all_df.to_csv(all_out, index=False)

num_cols = [c for c in all_df.columns if c not in ["space","fold"] and pd.api.types.is_numeric_dtype(all_df[c])]
agg = pd.DataFrame({
    "metric": num_cols,
    "mean": [all_df[c].mean() for c in num_cols],
    "std":  [all_df[c].std() for c in num_cols],
})
agg["mean_std"] = agg["mean"].map(lambda x: f"{x:.6g}") + " ± " + agg["std"].map(lambda x: f"{x:.3g}")
agg_out = OUT_ROOT / f"{BASELINE}_all_metrics_aggregated.csv"
agg.to_csv(agg_out, index=False)

print("Wrote:", all_out)
print("Wrote:", agg_out)
PY